# Vivarium PoC — MISM Execution Platform

This notebook demonstrates a minimal vivarium-core simulation executed
headlessly by the MISM execution platform. It defines a simple
transcription process, runs it, and writes results to confirm
end-to-end execution.

In [ ]:
import json
import os

from vivarium.core.process import Process
from vivarium.core.engine import Engine
from vivarium.core.composer import Composer

print('vivarium-core imported successfully')

## 1. Define a Transcription Process

A simple deterministic transcription model: DNA produces mRNA at a
constant rate proportional to the gene copy number.

In [ ]:
class Transcription(Process):
    defaults = {
        'transcription_rate': 0.01,
    }

    def ports_schema(self):
        return {
            'cell': {
                'DNA': {
                    '_default': 1.0,
                    '_emit': True,
                },
                'mRNA': {
                    '_default': 0.0,
                    '_emit': True,
                },
            }
        }

    def next_update(self, timestep, states):
        dna = states['cell']['DNA']
        rate = self.parameters['transcription_rate']
        new_mrna = dna * rate * timestep
        return {
            'cell': {
                'mRNA': new_mrna,
            }
        }

print('Transcription process defined')

## 2. Run the Simulation

In [ ]:
tx = Transcription({'transcription_rate': 0.05})
composite = tx.generate()

engine = Engine(composite=composite)
engine.update(100.0)

data = engine.emitter.get_data()
times = sorted(data.keys())

print(f'Simulation complete: {len(times)} timepoints')
print(f'  t=0   mRNA = {data[times[0]]["cell"]["mRNA"]:.2f}')
print(f'  t={times[-1]}  mRNA = {data[times[-1]]["cell"]["mRNA"]:.2f}')

## 3. Write Output

Write simulation results to `/output` (mounted volume) if available,
otherwise to the current directory.

In [ ]:
output_dir = os.environ.get('OUTPUT_PATH', '/output')
if not os.path.exists(output_dir):
    output_dir = '.'

results = {
    'timepoints': len(times),
    'final_time': times[-1],
    'final_mRNA': data[times[-1]]['cell']['mRNA'],
    'timeseries': [
        {'time': t, 'mRNA': data[t]['cell']['mRNA'], 'DNA': data[t]['cell']['DNA']}
        for t in times
    ],
}

output_path = os.path.join(output_dir, 'results.json')
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results written to {output_path}')
print(f'Execution complete.')